In [1]:
!pip install requests pandas matplotlib seaborn

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import requests
import pandas as pd 

BASE_URL = "http://localhost:8000/api/v1"

try:
    requests.post(f"{BASE_URL}/auth/register", json={
        "email": "ejemplo@test.com",
        "password": "secreto123",
        "full_name": "Usuario ejemplo"
    })
except:
    pass

resp = requests.post(f"{BASE_URL}/auth/login", json={
    "email": "demo@test.com",
    "password": "demo1234"
})
token = resp.json()["access_token"]
headers = {"Authorization": f"Bearer {token}"}
print("Token obtenido")

# login_resp = requests.post(f"{BASE_URL}/auth/login", json={
#     "email": "demo@test.com",
#     "password": "demo1234"})

# token = login_resp.json()["access_token"]
# headers = {"Authorization": f"Bearer {token}"}
# print("Token obtenido")

Token obtenido


In [3]:
muestras = [
    [51., 3.5, 1.4, 0.2],
    [6.0, 2.9, 4.5, 1.5],
    [5.5, 2.4, 3.8, 1.1],
    [7.7, 3.0, 6.1, 2.3]
]

resultados = []
for features in muestras:
    resp = requests.post(f"{BASE_URL}/predictions/predict", json={"features": features}, headers=headers)
    resp.raise_for_status()
    pred = resp.json()
    resultados.append(pred)
    print(f"Features: {features} -> Clase: {pred['predicted_class']} "
          f"(Probabilities: {pred['probability']:.4f}, name: {pred.get('class_name')})")

df_pred = pd.DataFrame(resultados)
df_pred

HTTPError: 500 Server Error: Internal Server Error for url: http://localhost:8000/api/v1/predictions/predict

In [ ]:
resp_hist = requests.get(f"{BASE_URL}/predictions/history?limit=10000", headers=headers)
resp_hist.raise_for_status()
datos_hist = resp_hist.json()

df_hist = pd.DataFrame(datos_hist)
if "created_at" in df_hist.columns:
    df_hist["created_at"] = pd.to_datetime(df_hist["created_at"])
print(f"Total de predicciones almacenadas: {len(df_hist)}")

df_hist.to_csv("historial_predicciones.csv", index=False)
df_hist.head(10)

Total de predicciones almacenadas: 1204


,predicted_class,probability,class_name,created_at,input_data
0,2,0.985833,None,2026-07-01 19:06:45.969346+00:00,"{'features': [7.7, 3.0, 6.1, 2.3]}"
1,1,0.942888,None,2026-07-01 19:06:45.948380+00:00,"{'features': [5.5, 2.4, 3.8, 1.1]}"
2,1,0.759735,None,2026-07-01 19:06:45.925693+00:00,"{'features': [6.0, 2.9, 4.5, 1.5]}"
3,1,0.999968,None,2026-07-01 19:06:45.868030+00:00,"{'features': [51.0, 3.5, 1.4, 0.2]}"
4,1,0.986634,None,2026-07-01 18:34:40.266206+00:00,"{'features': [7.7, 2.7, 1.1, 1.6]}"
5,2,0.943050,None,2026-07-01 18:34:40.180301+00:00,"{'features': [6.1, 2.3, 4.6, 2.3]}"
6,1,0.900817,None,2026-07-01 18:34:40.094449+00:00,"{'features': [7.4, 4.0, 3.9, 1.1]}"
7,1,0.965045,None,2026-07-01 18:34:40.009995+00:00,"{'features': [6.5, 2.4, 1.4, 1.1]}"
8,1,0.912394,None,2026-07-01 18:34:39.929479+00:00,"{'features': [5.5, 2.0, 4.1, 1.2]}"
9,1,0.992965,None,2026-07-01 18:34:39.843581+00:00,"{'features': [7.6, 2.6, 2.0, 1.1]}"


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.datasets import load_iris
import pandas as pd
import joblib

iris = load_iris(as_frame=True).frame

X = iris.drop(columns="target")
y = iris["target"]

X_train, X_test, y_train, y_test, = train_test_split(X, y, test_size=0.2, random_state=42)

rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
target_names = load_iris().target_names
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred, target_names=target_names))

joblib.dump(rf, "random_forest_model.joblib")

Accuracy: 1.0000
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      1.00      1.00         9
   virginica       1.00      1.00      1.00        11

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30



['random_forest_model.joblib']

In [ ]:
df_hist.count()

predicted_class    1204
probability        1204
class_name            0
created_at         1204
input_data         1204
dtype: int64